# Kafka za streaming slika
Koristi zookeeper, kafka samo prima/ šalje ID-eve slika iz bucketa zbog optimizacije performansi

In [5]:
%%writefile ../kafka.docker-compose.yml
version: '3'

networks:
  cv-network:
    #name: "app-network"
    #driver: bridge  #TODO: remove this line after merging with docker-compose.yml
    external: true

volumes:
  kafka-data:
  kafka-secrets:
  zookeeper-data:
  zookeeper-log:
  zookeeper-secrets:

services:
  # --- Infrastructure Services (largely unchanged) ---

  
  zookeeper:
    image:  confluentinc/cp-zookeeper:latest
    container_name: zookeeper
    stop_grace_period: 30s 
    networks:
      - cv-network
    env_file:
        - ./Kafka/zookeeper.env
      
    volumes:
      - zookeeper-data:/var/lib/zookeeper/data
      - zookeeper-log:/var/lib/zookeeper/log
      - zookeeper-secrets:/etc/zookeeper/secrets


  kafka:
    image: confluentinc/cp-kafka:latest
    container_name: kafka
    networks:
      - cv-network
    depends_on:
      - zookeeper
    ports:
      - "9092:9092"
    env_file:
        - ./Kafka/kafka.env
      
    volumes:
      - kafka-data:/var/lib/kafka/data
      - kafka-secrets:/etc/kafka/secrets

Overwriting ../kafka.docker-compose.yml


# Env datoteke za konfiguriranje kontenjera

In [7]:
%%writefile ./kafka.env      
KAFKA_BROKER_ID=1
KAFKA_ZOOKEEPER_CONNECT=zookeeper:2181
KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://kafka:29092,PLAINTEXT_HOST://10.124.17.151:9092
KAFKA_LISTENERS=PLAINTEXT://0.0.0.0:29092,PLAINTEXT_HOST://0.0.0.0:9092
KAFKA_LISTENER_SECURITY_PROTOCOL_MAP=PLAINTEXT:PLAINTEXT,PLAINTEXT_HOST:PLAINTEXT
KAFKA_INTER_BROKER_LISTENER_NAME=PLAINTEXT
KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR=1

Overwriting ./kafka.env


In [4]:
%%writefile ./zookeeper.env
ZOOKEEPER_CLIENT_PORT=2181
ZOOKEEPER_TICK_TIME=2000

Overwriting ./zookeeper.env


# Modifikacija .env datoteke i dodavanje novog compose file-a na kraj

In [2]:
import os

def update_compose_file_env(env_file_path, compose_file_name):
    """
    Updates the .env file to include the specified compose file in the COMPOSE_FILE variable.

    Args:
        env_file_path (str): The path to the .env file.
        compose_file_name (str): The name of the compose file to add.
    """
    try:
        with open(env_file_path, 'r') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"Error: .env file not found at {env_file_path}")
        return

    updated = False
    with open(env_file_path, 'w') as f:
        for line in lines:
            if line.startswith("COMPOSE_FILE="):
                if compose_file_name not in line:
                    line = line.strip()
                    if line.endswith("="):
                        line += compose_file_name + "\n"
                    else:
                        line += ":" + compose_file_name + "\n"
                    updated = True
            f.write(line)

    if updated:
        print(f"Updated COMPOSE_FILE in {env_file_path}")
    else:
        print(f"COMPOSE_FILE already contains {compose_file_name} in {env_file_path}")

if __name__ == "__main__":
    env_file = "../.env"
    compose_file = "kafka.docker-compose.yml"
    update_compose_file_env(env_file, compose_file)

COMPOSE_FILE already contains kafka.docker-compose.yml in ../.env


# Setting up topic for images (run this after services are running)

Create dockerfile for admin container

In [1]:
%%writefile topic_creator/Dockerfile
FROM python:3.9-slim

WORKDIR /app

COPY requirements.txt . 
RUN pip install --no-cache-dir -r requirements.txt

COPY topic_creator.py .

CMD ["python", "topic_creator.py"]

Writing topic_creator/Dockerfile


Requirements for Python script

In [2]:
%%writefile topic_creator/requirements.txt
kafka-python

Writing topic_creator/requirements.txt


Topic creator code

In [4]:
%%writefile topic_creator/topic_creator.py
import os
import time
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError, NoBrokersAvailable

# Kafka Configuration
KAFKA_BOOTSTRAP_SERVERS = os.environ.get('KAFKA_BOOTSTRAP_SERVERS', 'kafka:29092') # Internal Docker address
TOPIC_NAME = os.environ.get('TOPIC_NAME', 'cctv-image-events')
NUM_PARTITIONS = int(os.environ.get('NUM_PARTITIONS', 3))
REPLICATION_FACTOR = int(os.environ.get('REPLICATION_FACTOR', 1))

# --- Wait for Kafka ---
def wait_for_kafka(bootstrap_servers, retries=10, delay=5):
    print(f"Waiting for Kafka at {bootstrap_servers}...")
    for i in range(retries):
        try:
            admin_client = KafkaAdminClient(bootstrap_servers=bootstrap_servers)
            # Trying to list topics is a good way to check if Kafka is truly ready
            admin_client.list_topics()
            print("Kafka is available.")
            return admin_client
        except NoBrokersAvailable:
            print(f"Kafka not available yet (attempt {i+1}/{retries}). Retrying in {delay}s...")
            time.sleep(delay)
        except Exception as e:
            print(f"An unexpected error occurred while waiting for Kafka: {e}")
            # Might indicate a different startup issue
            return None # Exit if wait fails for other reasons

    print("Kafka did not become available within the timeout.")
    return None


# --- Create Topic ---
if __name__ == "__main__":
    admin_client = None
    try:
        # Use the internal Docker network address for bootstrap servers
        admin_client = wait_for_kafka(KAFKA_BOOTSTRAP_SERVERS.split(','))

        if admin_client:
            topic_list = [
                NewTopic(
                    name=TOPIC_NAME,
                    num_partitions=NUM_PARTITIONS,
                    replication_factor=REPLICATION_FACTOR
                )
            ]
            print(f"Attempting to create topic: {TOPIC_NAME} with {NUM_PARTITIONS} partitions and replication factor {REPLICATION_FACTOR}")

            try:
                admin_client.create_topics(
                    new_topics=topic_list,
                    validate_only=False
                )
                print(f"Topic '{TOPIC_NAME}' created successfully (or already exists).")
            except TopicAlreadyExistsError:
                print(f"Topic '{TOPIC_NAME}' already exists.")
            except Exception as e:
                print(f"Error creating topic '{TOPIC_NAME}': {e}")
                exit(1) # Exit with error code on failure

    except Exception as e:
        print(f"An error occurred during setup: {e}")
        exit(1)
    finally:
        if admin_client:
            admin_client.close()
            print("Admin client closed.")

Overwriting topic_creator/topic_creator.py


Docker-compose service for kafka admin

In [9]:
%%writefile topic_creator/topic-creator.kafka.admin.yml

version: '3'

networks:
    cv-network:
        external: true

services:
  kafka-topic-creator:
    build:
      context: . #Docker file is in same folder as Dockerfile
    container_name: kafka-topic-creator
    environment:
      KAFKA_BOOTSTRAP_SERVERS: kafka:29092 # Connect to Kafka using its service name and internal port
      TOPIC_NAME: cctv-image-events
      NUM_PARTITIONS: 3
      REPLICATION_FACTOR: 1
    
    networks:
        - cv-network
    
    

Overwriting topic_creator/topic-creator.kafka.admin.yml


Run the admin container once to define channel

In [10]:
!docker compose -f ./topic_creator/topic-creator.kafka.admin.yml run --rm kafka-topic-creator

WARN[0000] /home/benjamin/Documents/ParkMan/Kafka/topic_creator/topic-creator.kafka.admin.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion 
Compose can now delegate builds to bake for better performance.
 To do so, set COMPOSE_BAKE=true.
[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/2)                                          docker:default
 => [kafka-topic-creator internal] load build definition from Dockerfile   0.0s
 => => transferring dockerfile: 212B                                       0.0s
 => [kafka-topic-creator internal] load metadata for docker.io/library/py  0.2s
[+] Building 0.3s (1/2)                                          docker:default
 => [kafka-topic-creator internal] load build definition from Dockerfile   0.0s
 => => transferring dockerfile: 212B                                       0.0s
 => [kafka-topic-creator internal] load metadata for docker.io/